In [1]:
import os
from jax import config
config.update("jax_enable_x64", True)

from petitRADTRANS.retrieval import Retrieval, RetrievalConfig
from petitRADTRANS.retrieval.models import gradient_profile_emission
from petitRADTRANS.retrieval.utils import gaussian_prior
from petitRADTRANS import physical_constants as cst


INFO:2026-05-14 15:22:18,807:jax._src.xla_bridge:810: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/anaconda3/envs/jaxprt/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/opt/anaconda3/envs/jaxprt/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


# Retrievals: Advanced features

Written by [Evert Nasedkin](mailto:nasedkinevert@gmail.com?subject=[petitRADTRANS]%20Retrievals).
Please cite pRT's retrieval package [(Nasedkin et al. 2024)](https://ui.adsabs.harvard.edu/abs/2024JOSS....9.5875N/abstract) in addition to pRT [(Mollière et al. 2019)](https://ui.adsabs.harvard.edu/abs/2019A%26A...627A..67M/abstract) if you make use of the retrieval package for your work.

Please make sure you thoroughly understand the retrieval setups described in the ["Basic Retrieval Tutorial"](retrieval_basic.html), the ["multiple data sets tutorial"](retrieval_multiple_data.html) and the ["emission spectrum retrieval tutorial"](retrieval_emission.html) before starting on this notebook. 

In this tutorial we will present some of the more advanced features of the retrieval package. Some of these are more fragile than the standard retrieval features, and may break unexpectedly. You should be comfortable writing your own model generating function for the retrieval, and be willing to do some debugging if something goes wrong. Caveats and warnings out of the way, let's get started. 

## Getting started
We'll start off using a similar set up to the ["emission spectrum retrieval tutorial"](retrieval_emission.html).

In [2]:
# Let's start by importing everything we need.
# To not have numpy start parallelizing on its own
os.environ["OMP_NUM_THREADS"] = "1"

# Import the class used to set up the retrieval.
# Import Prior functions, if necessary.
# Import atmospheric model function

In [3]:
# Define the pRT run setup
retrieval_config = RetrievalConfig(
    retrieval_name="HR8799e_example",  # give a useful name for your retrieval
    run_mode="retrieval",  # 'retrieve' to run, or 'evaluate' to make plots
    adaptive_mesh_refinement=True,  # adaptive mesh refinement, slower if True
    scattering_in_emission=True,  # add scattering for emission spectra clouds
    sampler_type='pymultinest'
)

For this example we include the GRAVITY data as published in Mollière et al. (2020). To reproduce the published results, please also include the archival SPHERE and GPI data from [Zurlo et al. (2015)](https://www.aanda.org/articles/aa/full_html/2016/03/aa26835-15/aa26835-15.html) and [Greenbaum et al. (2016)](https://iopscience.iop.org/article/10.3847/1538-3881/aabcb8).

In [4]:
# Read in Data

# Here we import petitRADTRANS to find the path of the example files on your machine
# In general this is not required, you just put the files in the folder that you are running
# Your script in, for example
import petitRADTRANS # need to get the name for the example data
path_to_data = "./"

retrieval_config.add_data(
    name='GRAVITY',
    path_to_observations=f"{path_to_data}/retrievals/emission/observations/HR8799e_Spectra.fits",
    data_resolution=500,
    model_resolution=1000,
    model_generating_function = gradient_profile_emission,
    scale_flux=True, # let the retrieval scale the flux by a multiplicative factor
    scale_uncertainties=True, # let the retrieval scale the flux uncertainties by a multiplicative factor
    fit_flux_offset=False, # Don't use an additive offset
    subtract_continuum=False, # Don't use continuum subtraction
    line_opacity_mode='c-k',
    radtrans_object=None, # if we wanted, we could direcly pass an existing Radtrans object here
    wavelengths=None, # likewise, rather than loading the data from file we could manually load it
)

## Photometric data

If we want to add photometry, we can do that as well! The photometry file should have the format:

`# Name, lower wavelength edge [um], upper wavelength edge [um], flux density [W/m2/micron], flux error [W/m2/micron]`

You are required to provide a model function for calculating the spectrum, as with spectral data, but also a photometric transformation function, which is used to convert the model spectrum into synthetic photometry. This would typically make use of the transmission function for a particular filter. We have reimplemented the the use of `SyntheticPhotometry` module of the `species` package (https://species.readthedocs.io/) to provide these functions automatically. If no function is provided, the RetrievalConfig will attempt to import `species` to use this module, using the `name` provided as the filter name.

If you are using transmission spectra, your photometric transformation function should model the difference between the clear and occulted stellar spectrum, returning the difference in (planet radius/stellar radius)^2.

In [5]:
path_to_data = "./"


In [6]:
retrieval_config.add_photometry(
    path_to_data + 'retrievals/emission/observations/hr8799e_photometry.txt',
    gradient_profile_emission,
    model_resolution=40,
    scale_flux=False,
    photometric_transformation_function=None,  # define the function to transform model spectrum to photometric points
)
# The photometric transformation function should be defined for each photometric band, which means we might need to
# call this function several times with different photometry files if you don't want to use the default functions,
# which are downloaded from the SVO Filter Profile Service using species.

/Users/nasedkin/python-packages/petitRADTRANS/petitRADTRANS/retrieval/data.py:381: UserWarning: obs file './retrievals/emission/observations/hr8799e_photometry.txt' should contain at most 4 columns (wavelength, wavelength bins (optional), flux, flux error), but has 5 column(s)
Additional columns will be ignored
  warnings.warn(
/Users/nasedkin/python-packages/petitRADTRANS/petitRADTRANS/retrieval/data.py:389: UserWarning: NANs present in obs file './retrievals/emission/observations/hr8799e_photometry.txt'
  warnings.warn(f"NANs present in obs file '{path}'")
/Users/nasedkin/python-packages/petitRADTRANS/petitRADTRANS/retrieval/data.py:381: UserWarning: obs file './retrievals/emission/observations/hr8799e_photometry.txt' should contain at most 4 columns (wavelength, wavelength bins (optional), flux, flux error), but has 5 column(s)
Additional columns will be ignored
  warnings.warn(
/Users/nasedkin/python-packages/petitRADTRANS/petitRADTRANS/retrieval/data.py:389: UserWarning: NANs pres

## Parameters and Priors

Here we add all of the parameters used in the retrieval for HR 8799 e, following the prescription of Mollière 2020.
There are many other approaches we could take: varying the temperature structure parameterisation, retrieving different cloud properties, adding in a blackbody CPD and so on.
It is highly recommended to look at the API documentation of `models.py`, or at the [Retrieval Models Tutorial](retrieval_models.html) to get a better idea of what options are available.

In [7]:
# Add parameters, and priors for free parameters
# This run uses the model of Molliere (2020) for HR8799e
# The lambda function provide uniform priors

# Distance to the planet in cm
retrieval_config.add_parameter(
    name='system_distance',
    is_free_parameter=False,
    value=41.2925 * cst.pc
)

# Log of the surface gravity in cgs units.
retrieval_config.add_parameter(
    'log_g',
    True,
    transform_prior_cube_coordinate=lambda x: 2 + 3.5 * x
)

# Planet radius in cm
retrieval_config.add_parameter(
    'planet_radius',
    True,
    transform_prior_cube_coordinate=lambda x: (0.7 + 1.3 * x) * cst.r_jup_mean
)

# Temperature profile, based on Zhang+ 2023
n_layers = 10
retrieval_config.add_parameter('top_of_atmosphere_pressure', False, value=-6)
retrieval_config.add_parameter('N_layers', False, value=n_layers)
retrieval_config.add_parameter('T_bottom', True,
                               transform_prior_cube_coordinate=lambda x: 2000.0 + 10000.0*x)

# dts = [0.07,0.10,0.18,0.27,0.24,0.25]
retrieval_config.add_parameter(
    'PTslope_1',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.25, 0.1)
)

retrieval_config.add_parameter(
    'PTslope_2',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.25, 0.1)
)

retrieval_config.add_parameter(
    'PTslope_3',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.26, 0.1)
)

retrieval_config.add_parameter(
    'PTslope_4',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.2, 0.1)
)

retrieval_config.add_parameter(
    'PTslope_5',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.12, 0.1)
)

retrieval_config.add_parameter(
    'PTslope_6',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.07, 0.1)
)

retrieval_config.add_parameter(
    'PTslope_7',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.0, 0.5)
)

retrieval_config.add_parameter(
    'PTslope_8',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.0, 0.5)
)

retrieval_config.add_parameter(
    'PTslope_9',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.0, 0.05)
)

retrieval_config.add_parameter(
    'PTslope_10',
    True,
    transform_prior_cube_coordinate=lambda x: gaussian_prior(x, 0.0, 0.05)
)


## Chemistry
There are multiple ways to parameterise the chemistry in a retrieval: we can rely on equilibrium chemistry, a simple parameterisation of disequilibrium chemistry, we can freely retrieve vertically constant abundances, we can vary the abundance profile with pressure, or we can use some combination of all of the above. Here we'll set up a typical disequilibrium chemistry retrieval, but allow CO to vary as a function of altitude.

In [8]:
# Chemistry
# A 'free retrieval' would have each line species as a parameter
# Using a (dis)equilibrium model, we only supply bulk parameters.  # noqa: E501
# Carbon quench pressure
retrieval_config.add_parameter(
    'log_pquench',
    True,
    transform_prior_cube_coordinate=lambda x: -6.0 + 9.0 * x
)
# Metallicity [Fe/H]
retrieval_config.add_parameter(
    'Fe/H',
    True,
    transform_prior_cube_coordinate=lambda x: -1.5 + 3.0 * x
)
# C/O ratio
retrieval_config.add_parameter(
    'C/O',
    True,
    transform_prior_cube_coordinate=lambda x: 0.1+1.5*x
)

In [9]:
retrieval_config.add_pressure_varying_line_species(
    'CO-NatAbund__HITEMP',
    # Use 'linear', 'cubic', or 'stepped' interpolation between nodes.
    mode='linear',
    # Fit node positions relative to the previous node in pressure space.
    pressure_spacing='relative',
    # Also fit the top and bottom abundances in addition to the nodes.
    n_nodes=3,
    free_mass_fraction_limits=(-7.0, 0.0),
    # Each pressure node varies relative to the node above it.
    log_pressure_range_prior=(0, 9),
)


In [10]:
# Clouds
# Based on an Ackermann-Marley (2001) cloud model
# Width of particle size distribution
retrieval_config.add_parameter(
    'sigma_lnorm',
    True,
    transform_prior_cube_coordinate=lambda x: 1.05 + 1.95 * x,
)

# Sedimentation parameter for each cloud species
retrieval_config.add_parameter(
    'fsed_MgSiO3(s)',
    True,
    transform_prior_cube_coordinate=lambda x: 10.0 * x,
)

retrieval_config.add_parameter(
    'fsed_Fe(s)',
    True,
    transform_prior_cube_coordinate=lambda x: 10.0 * x,
)

# Cloud particle radius for each cloud species
retrieval_config.add_parameter(
    'log_cloud_radius_MgSiO3(s)',
    True,
    transform_prior_cube_coordinate=lambda x: -7 + 10.0 * x,
)

retrieval_config.add_parameter(
    'log_cloud_radius_Fe(s)',
    True,
    transform_prior_cube_coordinate=lambda x: -7 + 10.0 * x,
)

# Cloud coverage fraction. Can be defined for all cloud species, or for a list of species.
retrieval_config.add_parameter(
    'cloud_coverage',
    True,
    transform_prior_cube_coordinate=lambda x: x,
)

retrieval_config.add_parameter(
    'patchy_clouds',
    False,
    ('MgSiO3(s)_crystalline_000__DHS',),

)

In [11]:
# Define opacity species to be included
retrieval_config.set_rayleigh_species(('H2', 'He'))
retrieval_config.set_continuum_opacities(('H2--H2', 'H2--He'))
retrieval_config.set_line_species(
    (
        'H2O__POKAZATEL',
        'CO-NatAbund',  # freely retrieving the abundance profile of CO
        'CH4__MM',
        'CO2',
        'HCN',
        'FeH',
        'H2S',
        'NH3',
        'PH3',
        'Na__NewAllard',
        'K__Allard',
        'TiO',
        'VO',
        'SiO',
    ),
    eq=False,
)

# Because we're allowing CO to vary with altitude, we need to remove it as a parameter
del retrieval_config.parameters["CO-NatAbund"]

/var/folders/66/46srd8053z5_wmg1tlyv021c0000gn/T/ipykernel_37286/3378884234.py:4: DeprecationWarning: The 'eq' argument is deprecated; use 'use_equilibrium_chemistry' instead.
  retrieval_config.set_line_species(


## Free cloud parameterisation
Rather than using the condensation curve to define where the clouds are located, we can directly retrieve the cloud base pressure and abundance.

In [12]:
retrieval_config.add_cloud_species(
    'Fe(s)_crystalline_000__DHS',
    use_equilibrium_chemistry=False,
    free_mass_fraction_limits=(-7.0, -1.0),
    cloud_base_pressure_limits=(-6, 3),
)

retrieval_config.add_cloud_species(
    'MgSiO3(s)_crystalline_000__DHS',
    use_equilibrium_chemistry=False,
    free_mass_fraction_limits=(-7.0, -1.0),
    cloud_base_pressure_limits=(-6, 3),
)

In [13]:
retrieval = Retrieval(
    retrieval_config,
    output_directory="",
    evaluate_sample_spectra=False
)

Setting up Radtrans object for data 'GRAVITY'...
Setting up AMR pressure grid.
Loading Radtrans opacities...
 Loading line opacities of species 'H2O__POKAZATEL.R1000' from file '/Users/nasedkin/python-packages/petitRADTRANS/input_data/opacities/lines/correlated_k/H2O/1H2-16O/1H2-16O__POKAZATEL.R1000_0.3-50mu.ktable.petitRADTRANS.h5'... Done.
 Loading line opacities of species 'CO-NatAbund.R1000' from file '/Users/nasedkin/python-packages/petitRADTRANS/input_data/opacities/lines/correlated_k/CO/C-O-NatAbund/C-O-NatAbund__HITEMP.R1000_0.1-250mu.ktable.petitRADTRANS.h5'... Done.
 Loading line opacities of species 'CH4__MM.R1000' from file '/Users/nasedkin/python-packages/petitRADTRANS/input_data/opacities/lines/correlated_k/CH4/12C-1H4/12C-1H4__MM.R1000_0.3-50mu.ktable.petitRADTRANS.h5'... Done.
 Loading line opacities of species 'CO2.R1000' from file '/Users/nasedkin/python-packages/petitRADTRANS/input_data/opacities/lines/correlated_k/CO2/12C-16O2/12C-16O2__UCL-4000.R1000_0.3-50mu.ktabl

In [14]:
# Before we run the retrieval, let's set up plotting.

for parameter in retrieval_config.parameters:
    retrieval_config.parameters[parameter].plot_in_corner = True

In [15]:
retrieval = Retrieval(
    retrieval_config,
    output_directory="./",
    evaluate_sample_spectra=False
)

Using provided Radtrans object for data 'GRAVITY'...
Using provided Radtrans object for data 'Keck/NIRC2.Ks'...
Using provided Radtrans object for data 'Paranal/NACO.Lp'...
Using provided Radtrans object for data 'Paranal/NACO.NB405'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_B_J'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_H23_2'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_H23_3'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_K12_1'...
Using provided Radtrans object for data 'Paranal/SPHERE.IRDIS_D_K12_2'...


As mentioned at the beginning of this tutorial, this retrieval is very complex and thus takes several days to run, with hundred of cores on a cluster.

To try to run the retrieval anyway, set `run_retrieval` below to `True`, then execute the cells below.

In [16]:
run_retrieval = False

In [17]:
if run_retrieval:
    retrieval.run(
        sampling_efficiency=0.8,
        const_efficiency_mode=False,
        n_live_points=400,
        log_z_convergence=0.5,
        n_iter_before_update=50,
        resume=False,
        max_iter=0,
        error_checking=True,
        force_serial_error_checking=False,
        seed=12345  # should be -1 for a real retrieval, defined here for reproducibility
    )

Once the retrieval is complete, the easiest way to generate standard output plots is to use the `plot_all` function.

In [18]:
if run_retrieval:
    retrieval.plot_all(contribution=True)

**Contact**

If you need any additional help, don't hesitate to contact [Evert Nasedkin](mailto:nasedkinevert@gmail.com?subject=[petitRADTRANS]%20Retrievals).